# Fase 2 — Análise exploratória e seleção de variáveisEste notebook gera a **Tabela 2** (estatísticas descritivas e zeros), a**Figura 2** (correlação de Spearman) e define quais features seguem parao pré-processamento.Usamos só os **644 municípios que têm nota do IEGM**, que são os que vão paraa clusterização. A capital fica fora porque é fiscalizada pelo TCM-SP, e nãopelo TCE-SP, e por isso não tem IEGM.

In [ ]:
import sysfrom pathlib import Pathimport numpy as npimport pandas as pdRAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()sys.path.insert(0, str(RAIZ / "src"))from config import (BASE_FINAL_CSV, DICIONARIO_CSV, DATA_PROCESSED,                    POPULACAO_MINIMA, LIMIAR_REDUNDANCIA)from estilo import aplicar_estilo, salvarfrom figuras import (plot_hist_populacao, plot_boxplot_taxas,                     plot_distribuicoes_log1p, plot_spearman)aplicar_estilo()print("Python:", sys.executable)   # conferir se é o Python que tem as bibliotecasbase = pd.read_csv(BASE_FINAL_CSV, dtype={"codigo_ibge": str})dic = pd.read_csv(DICIONARIO_CSV)print(f"base_final: {base.shape[0]} municípios x {base.shape[1]} colunas")

A lista de features vem do `dicionario_base.csv`. Assim, se alguém mudar opapel de uma coluna no `merge_bases.py`, o notebook acompanha sem precisareditar nada aqui.

In [ ]:
bloco_de = dic.set_index("coluna")["bloco"]ORDEM_BLOCOS = ["criminalidade", "socioeconomico", "gestao"]features = sorted(dic.loc[dic["papel"] == "feature", "coluna"],                  key=lambda c: (ORDEM_BLOCOS.index(bloco_de[c]), c))taxas = [c for c in features if bloco_de[c] == "criminalidade"]   # pelo bloco, não pelo prefixoordinais = [c for c in features if bloco_de[c] == "gestao"]modelagem = base[~base["flag_sem_iegm"]].copy()print(f"{len(features)} features em {len(modelagem)} municípios:")print(pd.Series([bloco_de[c] for c in features]).value_counts().reindex(ORDEM_BLOCOS).to_string())

## 0. Primeiro contato com a baseAntes de transformar qualquer coisa: o que tem na base, o que está faltandoe como as variáveis se distribuem. Nesta seção não tomamos nenhuma decisãode modelagem.

In [ ]:
base.info()

In [ ]:
ausentes = base.isna().sum()ausentes[ausentes > 0]

Os valores ausentes são de dois tipos. As colunas do IEGM faltam em um únicomunicípio, a capital. As duas proporções de veículo faltam nos municípiosque não tiveram nenhum veículo roubado ou furtado em três anos: semdenominador, não dá para calcular. Nenhum ausente é erro de coleta.

In [ ]:
for b in ORDEM_BLOCOS:    cols = [c for c in features if bloco_de[c] == b]    print(f"\n--- {b} ---")    display(modelagem[cols].describe().round(2).T)

O que dá para ver nas três tabelas: nas taxas criminais a média é sempremaior que a mediana, e o máximo fica muito longe do terceiro quartil (caudalonga para a direita). No bloco socioeconômico, o PIB per capita chega aR$ 580 mil, enquanto a renda domiciliar mediana não passa de R$ 2.500. NoIEGM, as notas ficam concentradas perto de 1.

In [ ]:
fig = plot_hist_populacao(base["populacao"], POPULACAO_MINIMA)salvar(fig, "figura_eda_populacao")

Quase um quarto dos municípios tem menos de 5.000 habitantes. Neles, umaúnica ocorrência em três anos já vira uma taxa alta por 100 mil habitantes.Por isso a base tem a coluna `flag_pop_pequena`.

In [ ]:
fig = plot_boxplot_taxas(modelagem, taxas)salvar(fig, "figura_eda_boxplot_taxas")

In [ ]:
def top10(coluna):    return (modelagem.nlargest(10, coluna)[["municipio", "populacao", coluna]]            .round(1).reset_index(drop=True))pd.concat([top10("taxa_cvli"), top10("taxa_roubo_outros")], axis=1,          keys=["taxa_cvli", "taxa_roubo_outros"])

Os dez maiores valores de CVLI são quase todos municípios pequenos (é oproblema dos números pequenos). Já os dez maiores de roubo são cidadesmédias e grandes. As duas listas são bem diferentes, o que já indica que astaxas não medem a mesma coisa.

In [ ]:
resumo_iegm = pd.DataFrame({    "valor mais comum": modelagem[ordinais].mode().iloc[0],    "% no valor mais comum": (modelagem[ordinais]                              .apply(lambda s: s.value_counts(normalize=True).max()) * 100).round(0),    "valores distintos": modelagem[ordinais].nunique(),})resumo_iegm

As notas do IEGM variam pouco. O caso mais forte é `i_planejamento_ord`: amaioria dos municípios tem exatamente o mesmo valor. Isso volta a aparecerno pré-processamento (notebook 03).

## 1. Tabela 2 — descritivas das taxas e zerosA coluna `% zeros` é a mais importante. Uma taxa com 30% de zeros misturamunicípios onde o crime não aconteceu com municípios onde aconteceu muito,e nenhuma transformação resolve isso.

In [ ]:
tabela2 = pd.DataFrame({    "média": modelagem[taxas].mean(),    "mediana": modelagem[taxas].median(),    "desvio": modelagem[taxas].std(),    "máx": modelagem[taxas].max(),    "% zeros": (modelagem[taxas] == 0).mean() * 100,    "assimetria": modelagem[taxas].skew(),}).round(2)tabela2.to_csv(DATA_PROCESSED / "tabela2_descritivas.csv")tabela2

## 2. Distribuições antes e depois do `log1p`Aplicamos `log1p` nas taxas criminais e nas duas variáveis em reais (PIB percapita e renda mediana), porque são as que têm cauda longa para a direita.Percentuais (que vão até 100) e notas de 1 a 5 ficam como estão.

In [ ]:
monetarias = ["pib_percapita", "renda_domiciliar_mediana"]log_cols = taxas + monetariasfig = plot_distribuicoes_log1p(modelagem, log_cols)salvar(fig, "figura_distribuicoes_log1p")

Depois do `log1p`, várias taxas ficam com assimetria negativa. Isso não éexagero da transformação: é o pico de zeros, que o `log1p` deixa em zeroenquanto o resto vai para perto de 3 ou 4. A tabela abaixo mostra issocalculando a assimetria sem os zeros.

In [ ]:
pd.DataFrame({    "% zeros": (modelagem[taxas] == 0).mean() * 100,    "assimetria bruta": modelagem[taxas].skew(),    "assimetria log1p": np.log1p(modelagem[taxas]).skew(),    "assimetria log1p sem zeros": [np.log1p(modelagem.loc[modelagem[c] > 0, c]).skew()                                   for c in taxas],}).round(2)

Sem os zeros, a assimetria fica perto de zero em todas as taxas. Ou seja, o`log1p` resolve a cauda longa e vamos manter. Os zeros são outro problema,tratado na análise de sensibilidade com `flag_pop_pequena`.

## 3. Figura 2 — correlação de SpearmanUsamos Spearman, e não Pearson, porque as taxas não têm distribuição normal.As variáveis estão em ordem de bloco, com linhas separando os blocos, paraver se os blocos se repetem ou se complementam. Só |ρ| ≥ 0,5 recebe o número.

In [ ]:
rho = modelagem[features].corr(method="spearman")fig = plot_spearman(rho, bloco_de, ORDEM_BLOCOS)salvar(fig, "figura2_spearman")

## 4. Poda por redundânciaRegra: pares com |ρ| acima de `LIMIAR_REDUNDANCIA` (0,85) medem a mesmacoisa, e um dos dois sai.

In [ ]:
pares = [(a, b, round(rho.loc[a, b], 3))         for i, a in enumerate(features) for b in features[i + 1:]         if abs(rho.loc[a, b]) > LIMIAR_REDUNDANCIA]if pares:    display(pd.DataFrame(pares, columns=["a", "b", "rho"]))else:    print(f"Nenhum par com |rho| > {LIMIAR_REDUNDANCIA}: as {len(features)} "          "features seguem para o pré-processamento.")maiores = rho.where(np.tril(np.ones_like(rho, dtype=bool), k=-1)).stack()maiores.abs().sort_values(ascending=False).head(8).round(3)

## 5. Conclusões1. **Nenhuma feature sai.** O par mais correlacionado é   `roubo_outros × roubo_veiculo`, abaixo do limiar. No bloco   socioeconômico, urbanização, lixo e esgoto se correlacionam entre 0,6 e   0,75, também abaixo do limiar.2. Na Figura 2, o quadrante criminalidade × gestão é quase branco. Os   blocos medem coisas diferentes, e é por isso que faz sentido juntá-los.3. `log1p` nas taxas e nas variáveis em reais fica mantido. Os zeros são um   problema à parte.4. Tabela 2 salva em `data/processed/tabela2_descritivas.csv` e Figura 2 em   `figuras/figura2_spearman.png`.